In [1]:
import oceanbench

oceanbench.__version__

'0.1.4'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# SPDX-FileCopyrightText: 2025 Mercator Ocean International <https://www.mercator-ocean.eu/>
#
# SPDX-License-Identifier: EUPL-1.2

# Open GLONET forecast sample with xarray
from datetime import datetime
import xarray

challenger_dataset: xarray.Dataset = (
    xarray.open_mfdataset(
        [
            "https://minio.dive.edito.eu/project-moi-glo36-oceanbench/public/20230104.zarr",
        ],
        engine="zarr",
        combine="nested",
        concat_dim="first_day_datetime",
        parallel=True,
    )
    .rename({"lat": "latitude", "lon": "longitude"})
    .unify_chunks()
    .assign({"first_day_datetime": [datetime.fromisoformat("2023-01-04")]})
)


challenger_dataset["zos"].attrs["standard_name"] = "sea_surface_height_above_geoid"
challenger_dataset["thetao"].attrs["standard_name"] = "sea_water_potential_temperature"
challenger_dataset["so"].attrs["standard_name"] = "sea_water_salinity"
challenger_dataset["uo"].attrs["standard_name"] = "eastward_sea_water_velocity"
challenger_dataset["vo"].attrs["standard_name"] = "northward_sea_water_velocity"
challenger_dataset["latitude"].attrs["standard_name"] = "latitude"
challenger_dataset["longitude"].attrs["standard_name"] = "longitude"

challenger_dataset

print(challenger_dataset)


<xarray.Dataset> Size: 50GB
Dimensions:             (first_day_datetime: 1, lead_day_index: 7, depth: 50,
                         latitude: 2041, longitude: 4320)
Coordinates:
  * depth               (depth) float32 200B 0.494 1.541 ... 5.275e+03 5.728e+03
  * latitude            (latitude) float32 8kB -80.0 -79.92 ... 89.92 90.0
  * lead_day_index      (lead_day_index) int64 56B 0 1 2 3 4 5 6
  * longitude           (longitude) float32 17kB -180.0 -179.9 ... 179.8 179.9
  * first_day_datetime  (first_day_datetime) datetime64[us] 8B 2023-01-04
Data variables:
    so                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 12GB dask.array<chunksize=(1, 1, 7, 256, 540), meta=np.ndarray>
    thetao              (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 12GB dask.array<chunksize=(1, 1, 7, 256, 540), meta=np.ndarray>
    uo                  (first_day_datetime, lead_day_index, depth, latitude, longitude) float32 12GB dask.arra

### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.171332,0.173314,0.175179,0.177530,0.169632,0.163417,0.164106
Temperature (°C) [sea_water_potential_temperature]{surface},0.704699,0.701466,0.700136,0.697572,0.699120,0.704456,0.707690
Salinity (PSU) [sea_water_salinity]{surface},0.583952,0.586672,0.588451,0.585134,0.587009,0.591588,0.595948
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.162184,0.164275,0.166836,0.169915,0.172704,0.175740,0.178907
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.162121,0.164976,0.167840,0.170580,0.173751,0.176348,0.179825
Temperature (°C) [sea_water_potential_temperature]{50m},0.983581,0.988462,0.989000,0.990270,1.000584,1.007591,1.015849
Salinity (PSU) [sea_water_salinity]{50m},0.323359,0.323532,0.323446,0.323274,0.323354,0.323476,0.323753
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.157921,0.159911,0.161535,0.163373,0.164937,0.166752,0.168619
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.156588,0.158985,0.160839,0.162257,0.163885,0.165606,0.167855
Temperature (°C) [sea_water_potential_temperature]{100m},1.067228,1.069467,1.074453,1.076334,1.082981,1.090051,1.097141


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},23.497942,23.253822,23.491993,23.841009,23.763325,24.005838,23.821915


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.172796,0.174528,0.177583,0.180769,0.182060,0.183417,0.183256
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.178164,0.181280,0.181264,0.182119,0.183277,0.185475,0.188725


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Message
0,Observation-based Class IV scores were not com...


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6
Lagrangian trajectory deviation (km) []{surface},14.796522,28.749208,41.791946,54.065723,65.840233


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.132830,0.135011,0.136306,0.135258,0.130190,0.126575,0.127505
Temperature (°C) [sea_water_potential_temperature]{surface},0.467767,0.476977,0.487375,0.500123,0.515669,0.538085,0.561382
Salinity (PSU) [sea_water_salinity]{surface},0.337515,0.343492,0.349799,0.354086,0.361679,0.373402,0.386378
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.140661,0.144091,0.148598,0.153858,0.159287,0.164414,0.171387
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.137320,0.142179,0.147499,0.152941,0.159396,0.164407,0.170601
Temperature (°C) [sea_water_potential_temperature]{50m},0.656582,0.676856,0.689798,0.704411,0.725821,0.748073,0.773979
Salinity (PSU) [sea_water_salinity]{50m},0.254716,0.256727,0.258867,0.260992,0.263284,0.265901,0.269062
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.135640,0.138914,0.142620,0.146709,0.150695,0.154597,0.158785
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.132145,0.136421,0.140054,0.143924,0.148400,0.152488,0.157080
Temperature (°C) [sea_water_potential_temperature]{100m},0.697711,0.713965,0.732609,0.755661,0.782602,0.812610,0.841695


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},20.060902,20.169046,20.695864,21.509706,22.390396,23.295109,23.937078


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.169131,0.172222,0.176603,0.181559,0.185345,0.186389,0.187139
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.175118,0.179872,0.180460,0.183230,0.187518,0.190938,0.193037


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6
Lagrangian trajectory deviation (km) []{surface},11.445717,22.282148,32.474842,42.342556,51.94735
